# F_S3.3 — Branch + LOWESS Mean-response Diagnostics

This notebook evaluates branch detection and, independently, the current
`case/caseA/lowess_classifier.py` mean-response classifier after MIC gating.
Each of the five KDE bandwidths independently completes local-mode detection and
cross-window tracking; the final label uses a 3/5 Branch majority vote, while 4–5
votes are retained as high confidence. It does not run dCor, Pearson routing,
power-law fitting, or SiZer.

**Candidate branch** is a detector output, never a ground-truth label.

Two diagnostic tables:

1. **F23/F24 (visible forks)** — expected labels are Branch / Uncertain /
   No Global Relationship, assigned from nominal SNR. Uncertain rows are shown
   but not scored.
2. **Other families (not forks)** — F25/F26 included as negatives, not as Branch.
   Expected is No branch when MIC ≥ 0.20, otherwise No Global Relationship.

Counts are shown with row percentages. A short TPR / FPR summary follows table 2.
The final section reports LOWESS response-strength and raw mean-shape diagnostics
by synthetic family, including a focused table for visible F23/F24 branches.

The expensive local multi-bandwidth KDE peak/valley summaries are cached in a
separate Parquet file. Changing downstream gates, fork metrics, or voting rules
therefore only reruns classification; set `REFIT_KDE=True` only after changing
the KDE bandwidths, window construction, or peak/valley extraction itself.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import clear_output, display
from tqdm.auto import tqdm


def locate_repo_root() -> Path:
    for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (root / 'D').is_dir() and (root / 'E').is_dir():
            return root
    raise FileNotFoundError('Could not locate the synthetic repository root.')


REPO_ROOT = locate_repo_root()
CASE_A_DIR = REPO_ROOT / 'case' / 'caseA'
if str(CASE_A_DIR) not in sys.path:
    sys.path.insert(0, str(CASE_A_DIR))

import importlib
import test_two_condition_kde
importlib.reload(test_two_condition_kde)
from test_two_condition_kde import (
    BANDWIDTH_FACTORS, classify_fitted_panel, fit_panel_kde, prepare_panel_xy,
)

# ============================================================
# 可调参数 — 修改这里即可改变分类行为
# ============================================================
PEARSON_THRESH = 0.60   # |ρ| >= 此值 → monotonic
MIC_MIN = 0.0           # MIC 门控（0.0 = 关闭）

SPREAD_PATTERN = 'constant'
X_DISTRIBUTION = 'even'
FORK_FAMILIES = ['F23', 'F24']
OTHER_FAMILIES = [
    'F01', 'F03', 'F05', 'F07', 'F13', 'F15',
    'F18', 'F19', 'F21', 'F22', 'F25', 'F26',
]
FAMILY_FILTER = FORK_FAMILIES + OTHER_FAMILIES
VARIANT_FILTER = ['mild', 'standard', 'strong']
REPEAT_FILTER = [1]
REFIT_KDE = True       # False: 读取已保存的局部 KDE；True: 强制重新拟合
REFIT_LOWESS = True     # LOWESS_FRAC 改变后重建一次；成功后可改回 False
KDE_MIN_WINDOW = 40
KDE_CACHE_VERSION = 1   # 窗口/KDE/峰谷提取算法改变时手动递增

INPUT_DIR = (
    REPO_ROOT / 'F' / 'output' / 'E' / 'E_S1'
    / SPREAD_PATTERN / 'selected' / X_DISTRIBUTION
)
METRICS_DIR = (
    REPO_ROOT / 'F' / 'output' / 'E' / 'E_S2'
    / SPREAD_PATTERN / 'selected' / X_DISTRIBUTION
)
OUTPUT_DIR = (
    REPO_ROOT / 'F' / 'output' / 'F_S3.3_caseA_classifier_test'
    / SPREAD_PATTERN / X_DISTRIBUTION
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
KDE_CACHE_PATH = OUTPUT_DIR / 'local_multimodal_kde.parquet'
PREDICTIONS_PATH = OUTPUT_DIR / 'branch_predictions.csv'

FAMILY_NAMES = {
    'F01': 'Linear positive',
    'F03': 'Power convex positive',
    'F05': 'Power concave positive',
    'F07': 'Saturation positive',
    'F13': 'S-curve positive',
    'F15': 'Threshold positive',
    'F18': 'Quadratic valley (U)',
    'F19': 'Spike',
    'F21': 'Cubic',
    'F22': 'Oscillation',
    'F23': 'Two Lines',
    'F24': 'Line and Parabola',
    'F25': 'Multi-regime threshold',
    'F26': 'Windowed threshold',
}

# Human-visible expected labels for fork families only.
# Candidate branch is reserved exclusively for classifier predictions.
VISIBLE_EXPECTED_RULES = {
    'F23': [
        (5.0, 'Branch'),
        (2.0, 'Branch'),
        (1.2, 'Uncertain'),
        (0.0, 'No Global Relationship'),
    ],
    'F24': [
        (2.0, 'Branch'),
        (1.5, 'Branch'),
        (1.2, 'Uncertain'),
        (0.0, 'No Global Relationship'),
    ],
}

PREDICTED_ORDER = [
    'Branch', 'Candidate branch', 'No branch', 'No Global Relationship',
]


def visible_expected(family_id: str, nominal_snr: float) -> str:
    snr = float(nominal_snr)
    for threshold, label in VISIBLE_EXPECTED_RULES[family_id]:
        if snr >= threshold:
            return label
    return 'No Global Relationship'


def assign_expected(family_id: str, nominal_snr: float, mic: float) -> str:
    if family_id in FORK_FAMILIES:
        return visible_expected(family_id, nominal_snr)
    if np.isfinite(mic) and mic >= MIC_MIN:
        return 'No branch'
    return 'No Global Relationship'


def diagnostic_with_rates(df: pd.DataFrame, expected_cats: list[str]) -> pd.DataFrame:
    """One row per family_id × observed expected label. Name is 1:1 with family."""
    work = df.copy()
    counts = (
        work.groupby(['family_id', 'expected'], sort=False)['predicted_group']
        .value_counts()
        .unstack(fill_value=0)
    )
    for col in PREDICTED_ORDER:
        if col not in counts.columns:
            counts[col] = 0
    counts = counts.reindex(columns=PREDICTED_ORDER, fill_value=0)
    n = counts.sum(axis=1).astype(int)
    counts = counts.loc[n > 0].copy()
    n = n.loc[counts.index]
    out = pd.DataFrame({'Expected N': n}, index=counts.index)
    for col in PREDICTED_ORDER:
        pct = counts[col].to_numpy() / n.to_numpy()
        out[col] = [f'{int(c)} ({p:.1%})' for c, p in zip(counts[col], pct)]
    out = out.reset_index()
    out.insert(1, 'name', out['family_id'].map(FAMILY_NAMES))
    fam_order = [fid for fid in FAMILY_FILTER if fid in set(out['family_id'])]
    out['family_id'] = pd.Categorical(out['family_id'], categories=fam_order, ordered=True)
    out['expected'] = pd.Categorical(out['expected'], categories=expected_cats, ordered=True)
    out = out.sort_values(['family_id', 'expected']).reset_index(drop=True)
    out['family_id'] = out['family_id'].astype(str)
    out['expected'] = out['expected'].astype(str)
    total = counts.sum(axis=0)
    total_n = int(total.sum())
    total_row = {
        'family_id': 'All',
        'name': '',
        'expected': '',
        'Expected N': total_n,
    }
    for col in PREDICTED_ORDER:
        p = (int(total[col]) / total_n) if total_n else 0.0
        total_row[col] = f'{int(total[col])} ({p:.1%})'
    out = pd.concat([out, pd.DataFrame([total_row])], ignore_index=True)
    return out.rename(columns={
        'family_id': 'Family',
        'name': 'Name',
        'expected': 'Expected label',
    })


def show_full(df: pd.DataFrame) -> None:
    with pd.option_context(
        'display.max_rows', None,
        'display.max_columns', None,
        'display.width', None,
        'display.max_colwidth', None,
        'display.expand_frame_repr', False,
    ):
        display(df)


cases = pd.read_csv(INPUT_DIR / 'cases.csv', low_memory=False)
cases = cases[
    cases['family_id'].isin(FAMILY_FILTER)
    & cases['variant_level'].isin(VARIANT_FILTER)
    & cases['repeat'].isin(REPEAT_FILTER)
].copy()
cases['point_index'] = pd.to_numeric(cases['candidate_index'], errors='raise').astype(np.int64)

metrics = pd.read_parquet(
    METRICS_DIR / 'metrics_full.parquet',
    columns=['case_id', 'MIC'],
)
data = cases.merge(metrics, on='case_id', how='inner', validate='one_to_one')
data['name'] = data['family_id'].map(FAMILY_NAMES)
data['expected'] = [
    assign_expected(fid, snr, mic)
    for fid, snr, mic in zip(data['family_id'], data['nominal_snr'], data['MIC'])
]

points = np.load(INPUT_DIR / 'scatter_points.npz')
x_all, y_all = points['x'], points['y']
del points

print(f'Loaded {len(data):,} cases')
print(f'PEARSON_THRESH = {PEARSON_THRESH}')
print(f'MIC_MIN = {MIC_MIN}')
print('Fork families :', FORK_FAMILIES)
print('Other families:', OTHER_FAMILIES)

/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 9,534 cases
PEARSON_THRESH = 0.6
MIC_MIN = 0.0
Fork families : ['F23', 'F24']
Other families: ['F01', 'F03', 'F05', 'F07', 'F13', 'F15', 'F18', 'F19', 'F21', 'F22', 'F25', 'F26']


In [2]:
KDE_CACHE_COLUMNS = [
    'case_id', 'cache_version', 'min_window', 'bandwidth_factor',
    'window_index', 'x_min', 'x_mid', 'x_max', 'y_median',
    'mode_low', 'mode_high', 'valley', 'median_valley_depth',
    'median_peak_valley_score', 'median_branch_mass',
    'median_mode_separation', 'median_peak_height_ratio',
    'n_bandwidth_pairs',
]


def _fit_to_cache_rows(case_id, fitted):
    rows = []
    for factor, window_points in fitted['bandwidth_points'].items():
        for window_index, point in enumerate(window_points):
            mode_low, mode_high = point['modes']
            rows.append({
                'case_id': str(case_id),
                'cache_version': KDE_CACHE_VERSION,
                'min_window': KDE_MIN_WINDOW,
                'bandwidth_factor': float(factor),
                'window_index': int(window_index),
                'x_min': point['x_min'],
                'x_mid': point['x_mid'],
                'x_max': point['x_max'],
                'y_median': point['y_median'],
                'mode_low': mode_low,
                'mode_high': mode_high,
                'valley': point['valley'],
                'median_valley_depth': point['median_valley_depth'],
                'median_peak_valley_score': point['median_peak_valley_score'],
                'median_branch_mass': point['median_branch_mass'],
                'median_mode_separation': point['median_mode_separation'],
                'median_peak_height_ratio': point['median_peak_height_ratio'],
                'n_bandwidth_pairs': point['n_bandwidth_pairs'],
            })
    return rows


def _restore_fitted_panel(case_rows, x, y):
    x_prepared, y_prepared = prepare_panel_xy(x, y)
    bandwidth_points = {float(f): [] for f in BANDWIDTH_FACTORS}
    ordered = case_rows.sort_values(['bandwidth_factor', 'window_index'])
    for row in ordered.itertuples(index=False):
        factor = float(row.bandwidth_factor)
        bandwidth_points[factor].append({
            'x_min': float(row.x_min),
            'x_mid': float(row.x_mid),
            'x_max': float(row.x_max),
            'y_median': float(row.y_median),
            'modes': (float(row.mode_low), float(row.mode_high)),
            'valley': float(row.valley),
            'median_valley_depth': float(row.median_valley_depth),
            'median_peak_valley_score': float(row.median_peak_valley_score),
            'median_branch_mass': float(row.median_branch_mass),
            'median_mode_separation': float(row.median_mode_separation),
            'median_peak_height_ratio': float(row.median_peak_height_ratio),
            'n_bandwidth_pairs': int(row.n_bandwidth_pairs),
            'bandwidth_factor': factor,
        })
    return {
        'x': x_prepared,
        'y': y_prepared,
        'bandwidth_points': bandwidth_points,
        'min_window': KDE_MIN_WINDOW,
    }


# ------------------------------------------------------------------
# Stage A: expensive local KDE fitting.  Reuse this Parquet cache.
# ------------------------------------------------------------------
if KDE_CACHE_PATH.exists() and not REFIT_KDE:
    kde_cache = pd.read_parquet(KDE_CACHE_PATH)
    kde_cache['case_id'] = kde_cache['case_id'].astype(str)
    versions = set(kde_cache['cache_version'].dropna().astype(int))
    min_windows = set(kde_cache['min_window'].dropna().astype(int))
    cached_factors = set(kde_cache['bandwidth_factor'].dropna().astype(float))
    expected_factors = set(map(float, BANDWIDTH_FACTORS))
    if (versions != {KDE_CACHE_VERSION}
            or min_windows != {KDE_MIN_WINDOW}
            or cached_factors != expected_factors):
        raise RuntimeError(
            'KDE cache configuration differs from the current detector. '
            'Set REFIT_KDE=True once to rebuild it.'
        )
    print(f'Loaded KDE cache: {KDE_CACHE_PATH} ({len(kde_cache):,} window-bandwidth rows)')
else:
    kde_cache = pd.DataFrame(columns=KDE_CACHE_COLUMNS)

eligible = data[pd.to_numeric(data['MIC'], errors='coerce').ge(MIC_MIN)].copy()
cached_ids = set(kde_cache['case_id'].astype(str)) if len(kde_cache) else set()
missing = eligible[~eligible['case_id'].astype(str).isin(cached_ids)]

if len(missing):
    print(f'Fitting local KDE for {len(missing):,} uncached cases ...')
    new_frames = []
    batch_rows = []
    for fit_i, row in enumerate(tqdm(
        missing.itertuples(index=False), total=len(missing),
        desc='Local multi-bandwidth KDE', leave=False,
    ), start=1):
        fitted = fit_panel_kde(
            x_all[int(row.point_index)],
            y_all[int(row.point_index)],
            min_window=KDE_MIN_WINDOW,
        )
        batch_rows.extend(_fit_to_cache_rows(row.case_id, fitted))
        if fit_i % 100 == 0:
            new_frames.append(pd.DataFrame(batch_rows, columns=KDE_CACHE_COLUMNS))
            batch_rows = []
    if batch_rows:
        new_frames.append(pd.DataFrame(batch_rows, columns=KDE_CACHE_COLUMNS))
    kde_cache = pd.concat([kde_cache, *new_frames], ignore_index=True)
    kde_cache = kde_cache.drop_duplicates(
        ['case_id', 'bandwidth_factor', 'window_index'], keep='last',
    ).sort_values(['case_id', 'bandwidth_factor', 'window_index'])
    kde_cache.to_parquet(KDE_CACHE_PATH, index=False, compression='zstd')
    print(f'Saved KDE cache: {KDE_CACHE_PATH} ({len(kde_cache):,} rows)')
elif not KDE_CACHE_PATH.exists():
    kde_cache.to_parquet(KDE_CACHE_PATH, index=False, compression='zstd')

kde_cache_indexed = kde_cache.set_index('case_id', drop=False).sort_index()


# ------------------------------------------------------------------
# Stage B: cheap downstream classification from the saved KDE layer.
# Rerun this stage after changing gates, fork metrics, or voting rules.
# ------------------------------------------------------------------
prediction_rows = []
for row in tqdm(
    data.itertuples(index=False),
    total=len(data),
    desc='Branch classification from KDE cache',
    leave=False,
):
    mic = float(row.MIC) if pd.notna(row.MIC) else np.nan
    if not np.isfinite(mic) or mic < MIC_MIN:
        predicted = 'No Global Relationship'
        branch_votes = np.nan
        support_votes = np.nan
        vote_pattern = None
        high_confidence = False
        support_median_separation = np.nan
    else:
        case_id = str(row.case_id)
        if case_id not in kde_cache_indexed.index:
            raise KeyError(f'Missing KDE cache rows for {case_id}')
        case_rows = kde_cache_indexed.loc[[case_id]].reset_index(drop=True)
        fitted = _restore_fitted_panel(
            case_rows,
            x_all[int(row.point_index)],
            y_all[int(row.point_index)],
        )
        result = classify_fitted_panel(fitted, coverage_mode='x_range')
        predicted = {
            'Branch': 'Branch',
            'Candidate': 'Candidate branch',
        }.get(result['status'], 'No branch')
        branch_votes = result.get('bandwidth_branch_votes', np.nan)
        support_votes = result.get('bandwidth_support_votes', np.nan)
        vote_pattern = result.get('bandwidth_vote_pattern')
        high_confidence = bool(result.get('high_confidence_branch', False))
        support_median_separation = result.get(
            'bandwidth_support_median_relative_mode_separation', np.nan,
        )
    prediction_rows.append({
        'case_id': row.case_id,
        'predicted_group': predicted,
        'branch_votes': branch_votes,
        'support_votes': support_votes,
        'vote_pattern': vote_pattern,
        'high_confidence_branch': high_confidence,
        'support_median_relative_mode_separation': support_median_separation,
    })

predictions = pd.DataFrame(prediction_rows)
predictions.to_csv(PREDICTIONS_PATH, index=False)
prediction_columns = [col for col in predictions.columns if col != 'case_id']
data = data.drop(columns=prediction_columns, errors='ignore')
data = data.merge(predictions, on='case_id', how='left', validate='one_to_one')

fork_data = data[data['family_id'].isin(FORK_FAMILIES)].copy()
fork_table = diagnostic_with_rates(
    fork_data,
    ['Branch', 'Uncertain', 'No Global Relationship'],
)

clear_output(wait=True)
print(f'KDE cache: {KDE_CACHE_PATH}')
print(f'Predictions: {PREDICTIONS_PATH}')
print('Table 1 — F23/F24 visible forks (row % of Expected N)')
print('Uncertain rows are shown but not scored.')
show_full(fork_table)


KDE cache: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/F/output/F_S3.3_caseA_classifier_test/constant/even/local_multimodal_kde.parquet
Predictions: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/F/output/F_S3.3_caseA_classifier_test/constant/even/branch_predictions.csv
Table 1 — F23/F24 visible forks (row % of Expected N)
Uncertain rows are shown but not scored.


,Family,Name,Expected label,Expected N,Branch,Candidate branch,No branch,No Global Relationship
0,F23,Two Lines,Branch,333,263 (79.0%),23 (6.9%),47 (14.1%),0 (0.0%)
1,F23,Two Lines,Uncertain,24,0 (0.0%),1 (4.2%),23 (95.8%),0 (0.0%)
2,F23,Two Lines,No Global Relationship,324,0 (0.0%),0 (0.0%),324 (100.0%),0 (0.0%)
3,F24,Line and Parabola,Branch,348,306 (87.9%),30 (8.6%),12 (3.4%),0 (0.0%)
4,F24,Line and Parabola,Uncertain,9,0 (0.0%),3 (33.3%),6 (66.7%),0 (0.0%)
5,F24,Line and Parabola,No Global Relationship,324,0 (0.0%),11 (3.4%),313 (96.6%),0 (0.0%)
6,All,,,1362,569 (41.8%),68 (5.0%),725 (53.2%),0 (0.0%)


## Table 2 — Other families (not forks)

F25 and F26 are negatives: overlapping / windowed thresholds are not two ridges
along \(y\) given \(x\). Expected is **No branch** if MIC ≥ 0.20, else
**No Global Relationship**. Candidate branch is still a prediction only.


In [3]:
other_data = data[data['family_id'].isin(OTHER_FAMILIES)].copy()
other_table = diagnostic_with_rates(
    other_data,
    ['No branch', 'No Global Relationship'],
)

print('Table 2 — other families (row % of Expected N)')
print('F25/F26 are included as Not Branch.')
show_full(other_table)

fork_pos = data[
    data['family_id'].isin(FORK_FAMILIES)
    & (data['expected'] == 'Branch')
]
other_neg = data[
    data['family_id'].isin(OTHER_FAMILIES)
    & (data['expected'] == 'No branch')
]


def _rate(series: pd.Series, value) -> float:
    if len(series) == 0:
        return float('nan')
    if isinstance(value, (list, tuple, set)):
        return float(series.isin(value).mean())
    return float((series == value).mean())


def _pct(series: pd.Series, value) -> str:
    rate = _rate(series, value)
    return '' if pd.isna(rate) else f'{rate:.1%}'


print(
    '\nSummary — one table. Rate columns are predicted labels; '
    'Accuracy is whether that prediction is correct for the expected class.'
)
print('Uncertain rows are excluded. Candidate is never a ground-truth label.')
print(
    'Accuracy: expected Branch counts Branch or Candidate as correct; '
    'expected No branch counts only No branch as correct.'
)

pos_pred = fork_pos['predicted_group']
neg_pred = other_neg['predicted_group']
n_pos = int(len(fork_pos))
n_neg = int(len(other_neg))
n_all = n_pos + n_neg

acc_pos = _rate(pos_pred, ['Branch', 'Candidate branch'])
acc_neg = _rate(neg_pred, 'No branch')
if n_all:
    n_correct_loose = (
        int(pos_pred.isin(['Branch', 'Candidate branch']).sum())
        + int((neg_pred == 'No branch').sum())
    )
    n_correct_strict = (
        int((pos_pred == 'Branch').sum())
        + int((neg_pred == 'No branch').sum())
    )
    acc_overall_loose = n_correct_loose / n_all
    acc_overall_strict = n_correct_strict / n_all
else:
    acc_overall_loose = acc_overall_strict = float('nan')

all_pred = pd.concat([pos_pred, neg_pred], ignore_index=True)

summary = pd.DataFrame([
    {
        'Evaluation': 'Expected Branch (F23/F24, visible SNR)',
        'N': n_pos,
        'Predicted Branch': _pct(pos_pred, 'Branch'),
        'Predicted Branch or Candidate': _pct(
            pos_pred, ['Branch', 'Candidate branch'],
        ),
        'Predicted No branch': _pct(pos_pred, 'No branch'),
        'Accuracy': '' if pd.isna(acc_pos) else f'{acc_pos:.1%}',
    },
    {
        'Evaluation': 'Expected No branch (other families, MIC ≥ 0.20)',
        'N': n_neg,
        'Predicted Branch': _pct(neg_pred, 'Branch'),
        'Predicted Branch or Candidate': _pct(
            neg_pred, ['Branch', 'Candidate branch'],
        ),
        'Predicted No branch': _pct(neg_pred, 'No branch'),
        'Accuracy': '' if pd.isna(acc_neg) else f'{acc_neg:.1%}',
    },
    {
        'Evaluation': 'Overall (two rows above)',
        'N': n_all,
        'Predicted Branch': _pct(all_pred, 'Branch'),
        'Predicted Branch or Candidate': _pct(
            all_pred, ['Branch', 'Candidate branch'],
        ),
        'Predicted No branch': _pct(all_pred, 'No branch'),
        'Accuracy': (
            '' if pd.isna(acc_overall_loose) else f'{acc_overall_loose:.1%}'
        ),
    },
])
print(
    '\nOn the Branch row, Predicted Branch or Candidate is loose recall '
    'and Predicted No branch is the miss rate.'
)
print(
    'On the No branch row, Predicted Branch or Candidate is the false-positive '
    'rate and Predicted No branch is correct rejection.'
)
print(
    f'Overall Accuracy is loose '
    f'({"" if pd.isna(acc_overall_loose) else f"{acc_overall_loose:.1%}"})'
    f'; strict (Branch only on positives) is '
    f'{"" if pd.isna(acc_overall_strict) else f"{acc_overall_strict:.1%}"}.'
)
show_full(summary)


Table 2 — other families (row % of Expected N)
F25/F26 are included as Not Branch.


,Family,Name,Expected label,Expected N,Branch,Candidate branch,No branch,No Global Relationship
0,F01,Linear positive,No branch,681,0 (0.0%),0 (0.0%),681 (100.0%),0 (0.0%)
1,F03,Power convex positive,No branch,681,0 (0.0%),0 (0.0%),681 (100.0%),0 (0.0%)
2,F05,Power concave positive,No branch,681,0 (0.0%),0 (0.0%),681 (100.0%),0 (0.0%)
3,F07,Saturation positive,No branch,681,0 (0.0%),0 (0.0%),681 (100.0%),0 (0.0%)
4,F13,S-curve positive,No branch,681,0 (0.0%),0 (0.0%),681 (100.0%),0 (0.0%)
5,F15,Threshold positive,No branch,681,0 (0.0%),0 (0.0%),681 (100.0%),0 (0.0%)
6,F18,Quadratic valley (U),No branch,681,0 (0.0%),0 (0.0%),681 (100.0%),0 (0.0%)
7,F19,Spike,No branch,681,0 (0.0%),202 (29.7%),479 (70.3%),0 (0.0%)
8,F21,Cubic,No branch,681,0 (0.0%),0 (0.0%),681 (100.0%),0 (0.0%)
9,F22,Oscillation,No branch,681,0 (0.0%),0 (0.0%),681 (100.0%),0 (0.0%)



Summary — one table. Rate columns are predicted labels; Accuracy is whether that prediction is correct for the expected class.
Uncertain rows are excluded. Candidate is never a ground-truth label.
Accuracy: expected Branch counts Branch or Candidate as correct; expected No branch counts only No branch as correct.

On the Branch row, Predicted Branch or Candidate is loose recall and Predicted No branch is the miss rate.
On the No branch row, Predicted Branch or Candidate is the false-positive rate and Predicted No branch is correct rejection.
Overall Accuracy is loose (93.7%); strict (Branch only on positives) is 93.1%.


,Evaluation,N,Predicted Branch,Predicted Branch or Candidate,Predicted No branch,Accuracy
0,"Expected Branch (F23/F24, visible SNR)",681,83.6%,91.3%,8.7%,91.3%
1,"Expected No branch (other families, MIC ≥ 0.20)",8172,0.6%,6.1%,93.9%,93.9%
2,Overall (two rows above),8853,7.0%,12.6%,87.4%,93.7%


## LOWESS mean-response diagnostics

This section calls the current `case/caseA/lowess_classifier.py` directly. It evaluates MIC-eligible cases and keeps mean-response strength/shape independent from branch detection.

Table 6 reports each continuous diagnostic by family. Min–Max is the full observed range; P05–P95 is the robust central 90% range, and Median describes the typical case. Undefined metrics remain missing rather than being replaced by zero.


In [4]:
# Current LOWESS mean-response classifier + cached diagnostic tables
if str(REPO_ROOT / 'F') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'F'))
import lowess_mean_response_diagnostics as _lmrd
importlib.reload(_lmrd)
from lowess_mean_response_diagnostics import (
    run_lowess_diagnostics, _shape_accuracy_tables, _family_table,
    _pipeline_table,
)

lowess_eval, lowess_tables = run_lowess_diagnostics(
    data, x_all, y_all,
    case_a_dir=CASE_A_DIR,
    output_dir=OUTPUT_DIR,
    mic_min=MIC_MIN,
    family_order=FAMILY_FILTER,
    family_names=FAMILY_NAMES,
    fork_families=FORK_FAMILIES,
    refit_lowess=REFIT_LOWESS,
    pearson_thresh=PEARSON_THRESH,
    n_jobs=-1,
)

import lowess_classifier as _lc
_r2_weak = float(_lc.R2_WEAK)

_r2_fmt = {c: '{:.3f}' for c in ['Min','P05','P25','Median','P75','P95','Max']}
_acc_fmt = {
    'Coverage': '{:.1%}', 'Strict accuracy': '{:.1%}',
    'Resolved accuracy': '{:.1%}', 'Monotonic recall': '{:.1%}',
    'Non-monotonic recall': '{:.1%}', 'Balanced accuracy': '{:.1%}',
}

# ============================================================
# Standard + Strong 诊断表
# ============================================================
_ss = lowess_eval[lowess_eval['variant_level'].isin(['standard', 'strong'])].copy()
_ss_r2 = pd.to_numeric(_ss['r2'], errors='coerce')
_ss_strong = _ss[_ss_r2.ge(_r2_weak)].copy()

# R² range
_r2_rows = []
for fid in FAMILY_FILTER:
    g = _ss_r2[_ss['family_id'] == fid].dropna()
    if g.empty:
        continue
    _r2_rows.append({
        'Family': fid, 'Name': FAMILY_NAMES[fid], 'N': len(g),
        'Min': g.min(), 'P05': g.quantile(0.05), 'P25': g.quantile(0.25),
        'Median': g.median(),
        'P75': g.quantile(0.75), 'P95': g.quantile(0.95), 'Max': g.max(),
    })
print('Table 3 — R² range by family (standard + strong)')
show_full(pd.DataFrame(_r2_rows).style.format(_r2_fmt))

# Classification results — all families, all data
print('\nTable 4 — Classification results by family (standard + strong, all families)')
show_full(_family_table(_ss, FAMILY_FILTER, FAMILY_NAMES).style.format({'Median R2': '{:.3f}'}))

# Classification results — all families, R² >= threshold only
print(f'\nTable 4a — Classification results by family (standard + strong, R² >= {_r2_weak}, all families)')
show_full(_family_table(_ss_strong, FAMILY_FILTER, FAMILY_NAMES).style.format({'Median R2': '{:.3f}'}))

# Classification pipeline: N → Weak/R² → shape → sub-shape
print(f'\nTable 4b — Classification pipeline (standard + strong)')
show_full(_pipeline_table(_ss, _r2_weak, FAMILY_FILTER, FAMILY_NAMES))

print('\nTable 5 — Response strength × shape (standard + strong)')
show_full(pd.crosstab(_ss['strength'], _ss['primary_shape'], margins=True))

# Accuracy — 6a: all families, 6b: scored families only (each row = one family)
_, ss_all_family = _shape_accuracy_tables(
    _ss, _r2_weak, FAMILY_NAMES, family_order=FAMILY_FILTER)
_, ss_scored_family = _shape_accuracy_tables(
    _ss, _r2_weak, FAMILY_NAMES)
print(f'\nTable 6a — Shape accuracy by family (standard + strong, R² >= {_r2_weak}, |ρ| >= {PEARSON_THRESH}, all families)')
show_full(ss_all_family.style.format(_acc_fmt))
print(f'Table 6b — Shape accuracy by scored family (standard + strong, R² >= {_r2_weak}, |ρ| >= {PEARSON_THRESH})')
show_full(ss_scored_family.style.format(_acc_fmt))

Fitting LOWESS for 9,534 MIC-eligible cases ...
Saved LOWESS cache: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/F/output/F_S3.3_caseA_classifier_test/constant/even/lowess_mean_response_curves.parquet (9,534 cases)
Saved LOWESS features: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/F/output/F_S3.3_caseA_classifier_test/constant/even/lowess_mean_response_features.csv (9,534 cases)
Table 3 — R² range by family (standard + strong)


,Family,Name,N,Min,P05,P25,Median,P75,P95,Max
0,F23,Two Lines,454,0.016,0.019,0.035,0.174,0.282,0.424,0.424
1,F24,Line and Parabola,454,0.014,0.015,0.029,0.182,0.213,0.290,0.291
2,F01,Linear positive,454,0.021,0.022,0.053,0.616,0.985,0.998,1.000
3,F03,Power convex positive,454,0.011,0.012,0.053,0.605,0.984,0.997,1.000
4,F05,Power concave positive,454,0.009,0.012,0.046,0.629,0.979,0.993,0.997
5,F07,Saturation positive,454,0.024,0.028,0.073,0.651,0.965,0.996,1.000
6,F13,S-curve positive,454,0.012,0.015,0.051,0.658,0.988,0.998,1.000
7,F15,Threshold positive,454,0.012,0.017,0.066,0.636,0.961,0.991,1.000
8,F18,Quadratic valley (U),454,0.013,0.014,0.051,0.633,0.986,0.997,1.000
9,F19,Spike,454,-0.067,0.013,0.037,0.210,0.280,0.415,0.994



Table 4 — Classification results by family (standard + strong, all families)


,Family,Name,N,Median R2,Weak,Uncertain,Detectable,Flat,Mono up,Mono down,Non-monotonic,Unresolved shape
0,F23,Two Lines,454,0.174,353 (77.8%),101 (22.2%),0 (0.0%),0 (0.0%),354 (78.0%),16 (3.5%),84 (18.5%),0 (0.0%)
1,F24,Line and Parabola,454,0.182,454 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),346 (76.2%),0 (0.0%),108 (23.8%),0 (0.0%)
2,F01,Linear positive,454,0.616,194 (42.7%),31 (6.8%),229 (50.4%),0 (0.0%),380 (83.7%),0 (0.0%),74 (16.3%),0 (0.0%)
3,F03,Power convex positive,454,0.605,195 (43.0%),31 (6.8%),228 (50.2%),0 (0.0%),437 (96.3%),0 (0.0%),17 (3.7%),0 (0.0%)
4,F05,Power concave positive,454,0.629,191 (42.1%),33 (7.3%),230 (50.7%),0 (0.0%),409 (90.1%),0 (0.0%),45 (9.9%),0 (0.0%)
5,F07,Saturation positive,454,0.651,187 (41.2%),33 (7.3%),234 (51.5%),0 (0.0%),349 (76.9%),0 (0.0%),105 (23.1%),0 (0.0%)
6,F13,S-curve positive,454,0.658,188 (41.4%),32 (7.0%),234 (51.5%),0 (0.0%),423 (93.2%),0 (0.0%),31 (6.8%),0 (0.0%)
7,F15,Threshold positive,454,0.636,189 (41.6%),33 (7.3%),232 (51.1%),0 (0.0%),400 (88.1%),0 (0.0%),54 (11.9%),0 (0.0%)
8,F18,Quadratic valley (U),454,0.633,191 (42.1%),32 (7.0%),231 (50.9%),0 (0.0%),0 (0.0%),0 (0.0%),454 (100.0%),0 (0.0%)
9,F19,Spike,454,0.210,402 (88.5%),48 (10.6%),4 (0.9%),10 (2.2%),0 (0.0%),0 (0.0%),444 (97.8%),0 (0.0%)



Table 4a — Classification results by family (standard + strong, R² >= 0.35, all families)


...

primary_shape,Flat,Monotonic down,Monotonic up,Non-monotonic,All
strength,,,,,
Detectable,0,0,1617,458,2075
Uncertain,0,0,474,118,592
Weak,118,16,1750,1805,3689
All,118,16,3841,2381,6356



Table 6a — Shape accuracy by family (standard + strong, R² >= 0.35, |ρ| >= 0.6, all families)


,Family,Name,N (R2 eligible),Monotonic,Non-monotonic,Unresolved,Coverage,Strict accuracy,Resolved accuracy,Monotonic recall,Non-monotonic recall,Balanced accuracy
0,F23,Two Lines,101,101,0,0,100.0%,nan%,nan%,nan%,nan%,nan%
1,F01,Linear positive,260,260,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
2,F03,Power convex positive,259,259,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
3,F05,Power concave positive,263,263,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
4,F07,Saturation positive,267,266,1,0,100.0%,99.6%,99.6%,99.6%,nan%,99.6%
5,F13,S-curve positive,266,266,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
6,F15,Threshold positive,265,265,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
7,F18,Quadratic valley (U),263,0,263,0,100.0%,100.0%,100.0%,nan%,100.0%,100.0%
8,F19,Spike,52,0,52,0,100.0%,100.0%,100.0%,nan%,100.0%,100.0%
9,F21,Cubic,263,263,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%


Table 6b — Shape accuracy by scored family (standard + strong, R² >= 0.35, |ρ| >= 0.6)


,Family,Name,N (R2 eligible),Monotonic,Non-monotonic,Unresolved,Coverage,Strict accuracy,Resolved accuracy,Monotonic recall,Non-monotonic recall,Balanced accuracy
0,F01,Linear positive,260,260,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
1,F03,Power convex positive,259,259,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
2,F05,Power concave positive,263,263,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
3,F07,Saturation positive,267,266,1,0,100.0%,99.6%,99.6%,99.6%,nan%,99.6%
4,F13,S-curve positive,266,266,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
5,F15,Threshold positive,265,265,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
6,F18,Quadratic valley (U),263,0,263,0,100.0%,100.0%,100.0%,nan%,100.0%,100.0%
7,F19,Spike,52,0,52,0,100.0%,100.0%,100.0%,nan%,100.0%,100.0%
8,F21,Cubic,263,263,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
9,F22,Oscillation,260,0,260,0,100.0%,100.0%,100.0%,nan%,100.0%,100.0%


In [5]:
# F23 monotonic cases (standard+strong, R²>=0.35): scatter + LOWESS
import matplotlib.pyplot as plt

_N_F23_PLOTS = 20
_MAX_SCATTER = 3000

_f23_mono = _ss_strong[
    (_ss_strong['family_id'] == 'F23')
    & _ss_strong['lowess_shape'].isin(['monotonic_up', 'monotonic_down'])
].copy()
_f23_mono['point_index'] = _f23_mono['case_id'].map(
    data.drop_duplicates('case_id').set_index('case_id')['point_index']
)

_curve_cache = pd.read_parquet(OUTPUT_DIR / 'lowess_mean_response_curves.parquet')
_curve_lookup = {
    str(cid): (np.asarray(xs, dtype=float), np.asarray(ys, dtype=float))
    for cid, xs, ys in zip(_curve_cache['case_id'], _curve_cache['x_lowess'], _curve_cache['y_lowess'])
}

_f23_mono = _f23_mono.sort_values('r2')
if len(_f23_mono) > _N_F23_PLOTS:
    _pos = np.linspace(0, len(_f23_mono) - 1, _N_F23_PLOTS).round().astype(int)
    _chosen = _f23_mono.iloc[np.unique(_pos)]
else:
    _chosen = _f23_mono

_n = len(_chosen)
_ncols = 5
_nrows = int(np.ceil(_n / _ncols))
_rng = np.random.default_rng(99)

fig, axes = plt.subplots(_nrows, _ncols, figsize=(3.4 * _ncols, 3.2 * _nrows), squeeze=False)
for i, (_, row) in enumerate(_chosen.iterrows()):
    ax = axes[i // _ncols, i % _ncols]
    pi = int(row['point_index'])
    x = np.asarray(x_all[pi], dtype=float)
    y = np.asarray(y_all[pi], dtype=float)
    fin = np.isfinite(x) & np.isfinite(y)
    x, y = x[fin], y[fin]
    if len(x) > _MAX_SCATTER:
        keep = _rng.choice(len(x), _MAX_SCATTER, replace=False)
        x, y = x[keep], y[keep]
    ax.scatter(x, y, s=6, color='#6b7280', alpha=0.15, linewidths=0, rasterized=True)
    xs, ys = _curve_lookup[str(row['case_id'])]
    ax.plot(xs, ys, color='#d81b60', linewidth=2.0)
    rho = float(row['ls_abs_pearson']) if pd.notna(row['ls_abs_pearson']) else float('nan')
    ax.set_title(
        f"R²={row['r2']:.2f}  |ρ|={rho:.2f}\n"
        f"{row['lowess_shape2']}  SNR={row['nominal_snr']:.2g}",
        fontsize=8,
    )
    ax.grid(alpha=0.18, linewidth=0.6)
for j in range(i + 1, _nrows * _ncols):
    axes[j // _ncols, j % _ncols].set_visible(False)
fig.suptitle(
    f'F23 Two Lines — {_n} monotonic cases (standard+strong, R²≥{_r2_weak})',
    fontsize=13, y=1.01,
)
fig.tight_layout()
plt.show()
plt.close(fig)
print(f'Showing {_n} / {len(_f23_mono)} F23 monotonic cases')

Showing 20 / 101 F23 monotonic cases


/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_4529/3339219957.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ============================================================
# Mild 诊断表
# ============================================================
_mi = lowess_eval[lowess_eval['variant_level'] == 'mild'].copy()
_mi_r2 = pd.to_numeric(_mi['r2'], errors='coerce')
_mi_strong = _mi[_mi_r2.ge(_r2_weak)].copy()

# R² range
_r2_rows_mi = []
for fid in FAMILY_FILTER:
    g = _mi_r2[_mi['family_id'] == fid].dropna()
    if g.empty:
        continue
    _r2_rows_mi.append({
        'Family': fid, 'Name': FAMILY_NAMES[fid], 'N': len(g),
        'Min': g.min(), 'P05': g.quantile(0.05), 'P25': g.quantile(0.25),
        'Median': g.median(),
        'P75': g.quantile(0.75), 'P95': g.quantile(0.95), 'Max': g.max(),
    })
print('Table 7 — R² range by family (mild)')
show_full(pd.DataFrame(_r2_rows_mi).style.format(_r2_fmt))

# Classification results — all families, all data
print('\nTable 8 — Classification results by family (mild, all families)')
show_full(_family_table(_mi, FAMILY_FILTER, FAMILY_NAMES).style.format({'Median R2': '{:.3f}'}))

# Classification results — all families, R² >= threshold only
print(f'\nTable 8a — Classification results by family (mild, R² >= {_r2_weak}, all families)')
show_full(_family_table(_mi_strong, FAMILY_FILTER, FAMILY_NAMES).style.format({'Median R2': '{:.3f}'}))

# Classification pipeline: N → Weak/R² → shape → sub-shape
print(f'\nTable 8b — Classification pipeline (mild)')
show_full(_pipeline_table(_mi, _r2_weak, FAMILY_FILTER, FAMILY_NAMES))

print('\nTable 9 — Response strength × shape (mild)')
show_full(pd.crosstab(_mi['strength'], _mi['primary_shape'], margins=True))

# Accuracy — 10a: all families, 10b: scored families only (each row = one family)
_, mi_all_family = _shape_accuracy_tables(
    _mi, _r2_weak, FAMILY_NAMES, family_order=FAMILY_FILTER)
_, mi_scored_family = _shape_accuracy_tables(
    _mi, _r2_weak, FAMILY_NAMES)
print(f'\nTable 10a — Shape accuracy by family (mild, R² >= {_r2_weak}, |ρ| >= {PEARSON_THRESH}, all families)')
show_full(mi_all_family.style.format(_acc_fmt))
print(f'Table 10b — Shape accuracy by scored family (mild, R² >= {_r2_weak}, |ρ| >= {PEARSON_THRESH})')
show_full(mi_scored_family.style.format(_acc_fmt))

Table 7 — R² range by family (mild)


,Family,Name,N,Min,P05,P25,Median,P75,P95,Max
0,F23,Two Lines,227,0.024,0.025,0.045,0.436,0.717,0.725,0.726
1,F24,Line and Parabola,227,0.013,0.013,0.016,0.176,0.240,0.247,0.252
2,F01,Linear positive,227,0.016,0.016,0.039,0.622,0.986,0.998,1.000
3,F03,Power convex positive,227,0.016,0.018,0.066,0.631,0.985,0.998,1.000
4,F05,Power concave positive,227,0.036,0.040,0.111,0.698,0.988,0.998,0.999
5,F07,Saturation positive,227,0.017,0.017,0.047,0.626,0.986,0.998,1.000
6,F13,S-curve positive,227,0.013,0.013,0.037,0.618,0.986,0.998,1.000
7,F15,Threshold positive,227,0.009,0.010,0.047,0.631,0.985,0.997,0.999
8,F18,Quadratic valley (U),227,0.034,0.037,0.093,0.633,0.984,0.997,1.000
9,F19,Spike,227,0.010,0.011,0.051,0.600,0.688,0.734,1.000



Table 8 — Classification results by family (mild, all families)


,Family,Name,N,Median R2,Weak,Uncertain,Detectable,Flat,Mono up,Mono down,Non-monotonic,Unresolved shape
0,F23,Two Lines,227,0.436,106 (46.7%),25 (11.0%),96 (42.3%),0 (0.0%),179 (78.9%),0 (0.0%),48 (21.1%),0 (0.0%)
1,F24,Line and Parabola,227,0.176,227 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),183 (80.6%),0 (0.0%),44 (19.4%),0 (0.0%)
2,F01,Linear positive,227,0.622,97 (42.7%),15 (6.6%),115 (50.7%),18 (7.9%),175 (77.1%),0 (0.0%),34 (15.0%),0 (0.0%)
3,F03,Power convex positive,227,0.631,95 (41.9%),16 (7.0%),116 (51.1%),0 (0.0%),227 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%)
4,F05,Power concave positive,227,0.698,88 (38.8%),18 (7.9%),121 (53.3%),0 (0.0%),227 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%)
5,F07,Saturation positive,227,0.626,96 (42.3%),16 (7.0%),115 (50.7%),0 (0.0%),227 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%)
6,F13,S-curve positive,227,0.618,97 (42.7%),15 (6.6%),115 (50.7%),0 (0.0%),173 (76.2%),2 (0.9%),52 (22.9%),0 (0.0%)
7,F15,Threshold positive,227,0.631,96 (42.3%),15 (6.6%),116 (51.1%),0 (0.0%),212 (93.4%),0 (0.0%),15 (6.6%),0 (0.0%)
8,F18,Quadratic valley (U),227,0.633,93 (41.0%),18 (7.9%),116 (51.1%),0 (0.0%),0 (0.0%),0 (0.0%),227 (100.0%),0 (0.0%)
9,F19,Spike,227,0.600,95 (41.9%),19 (8.4%),113 (49.8%),0 (0.0%),0 (0.0%),0 (0.0%),227 (100.0%),0 (0.0%)



Table 8a — Classification results by family (mild, R² >= 0.35, all families)


...

primary_shape,Flat,Monotonic down,Monotonic up,Non-monotonic,All
strength,,,,,
Detectable,0,118,911,229,1258
Uncertain,0,17,137,37,191
Weak,18,94,1009,608,1729
All,18,229,2057,874,3178



Table 10a — Shape accuracy by family (mild, R² >= 0.35, |ρ| >= 0.6, all families)


,Family,Name,N (R2 eligible),Monotonic,Non-monotonic,Unresolved,Coverage,Strict accuracy,Resolved accuracy,Monotonic recall,Non-monotonic recall,Balanced accuracy
0,F23,Two Lines,121,121,0,0,100.0%,nan%,nan%,nan%,nan%,nan%
1,F01,Linear positive,130,130,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
2,F03,Power convex positive,132,132,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
3,F05,Power concave positive,139,139,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
4,F07,Saturation positive,131,131,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
5,F13,S-curve positive,130,130,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
6,F15,Threshold positive,131,131,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
7,F18,Quadratic valley (U),134,0,134,0,100.0%,100.0%,100.0%,nan%,100.0%,100.0%
8,F19,Spike,132,0,132,0,100.0%,100.0%,100.0%,nan%,100.0%,100.0%
9,F21,Cubic,134,134,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%


Table 10b — Shape accuracy by scored family (mild, R² >= 0.35, |ρ| >= 0.6)


,Family,Name,N (R2 eligible),Monotonic,Non-monotonic,Unresolved,Coverage,Strict accuracy,Resolved accuracy,Monotonic recall,Non-monotonic recall,Balanced accuracy
0,F01,Linear positive,130,130,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
1,F03,Power convex positive,132,132,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
2,F05,Power concave positive,139,139,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
3,F07,Saturation positive,131,131,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
4,F13,S-curve positive,130,130,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
5,F15,Threshold positive,131,131,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
6,F18,Quadratic valley (U),134,0,134,0,100.0%,100.0%,100.0%,nan%,100.0%,100.0%
7,F19,Spike,132,0,132,0,100.0%,100.0%,100.0%,nan%,100.0%,100.0%
8,F21,Cubic,134,134,0,0,100.0%,100.0%,100.0%,100.0%,nan%,100.0%
9,F22,Oscillation,135,135,0,0,100.0%,0.0%,0.0%,nan%,0.0%,0.0%


## LOWESS threshold sensitivity

The tables above use the current gates: Weak if \(R^2 < 0.35\), Detectable if \(R^2 \ge 0.60\), and monotonic if the EQ15 LOWESS \(|\rho| \ge\) `PEARSON_THRESH`. The next cell does not refit LOWESS. It only reapplies those numeric gates to the saved features.

Expected mean-shape for the Pearson sweep: monotonic for F01/F03/F05/F07/F13/F15/F21; non-monotonic for U (F18), Spike (F19), and Oscillation (F22). F25 is excluded from scoring. F23/F24/F26 are left unscored. Flat cases are kept as Flat and excluded from the Pearson percentages.

In [7]:
# Sensitivity of R2 and Pearson gates (no LOWESS refit)
from lowess_classifier import R2_DETECT, R2_WEAK

sens = lowess_eval.copy()
sens['r2'] = pd.to_numeric(sens['r2'], errors='coerce')
sens['MIC'] = pd.to_numeric(sens['MIC'], errors='coerce')
sens['ls_abs_pearson'] = pd.to_numeric(sens['ls_abs_pearson'], errors='coerce')
sens['ls_n_tp'] = pd.to_numeric(sens['ls_n_tp'], errors='coerce')

PEARSON_SWEEP = [0.50, 0.60, 0.70, 0.80]
R2_WEAK_SWEEP = [0.20, 0.30, 0.35, 0.40, 0.50]
R2_DETECT_SWEEP = [0.50, 0.55, 0.60, 0.70]
MIC_SWEEP = [0.20, 0.30, 0.40, 0.50, 0.60]
EXPECTED_NONMONO = {'F18', 'F19', 'F22'}
EXPECTED_MONO = {
    'F01', 'F03', 'F05', 'F07', 'F13', 'F15', 'F21',
}


def _pct_fmt(n, d):
    if d == 0:
        return '—'
    return f'{int(n)} ({n / d:.1%})'


def _family_block():
    return [fid for fid in FAMILY_FILTER if fid in set(sens['family_id'])]


# --- Pearson |rho| gate ---
pearson_rows = []
for family_id in _family_block():
    group = sens[sens['family_id'] == family_id]
    scored = group[group['lowess_shape'] != 'flat']
    row = {
        'Family': family_id,
        'Name': FAMILY_NAMES[family_id],
        'N': len(group),
        'N scored': len(scored),
        'Expected': (
            'Non-monotonic' if family_id in EXPECTED_NONMONO
            else 'Monotonic' if family_id in EXPECTED_MONO
            else 'Unscored'
        ),
    }
    for thresh in PEARSON_SWEEP:
        n_nm = int((scored['ls_abs_pearson'] < thresh).sum())
        mark = ' *' if abs(thresh - PEARSON_THRESH) < 1e-12 else ''
        row[f'Non-mono |ρ|<{thresh:.2f}{mark}'] = _pct_fmt(n_nm, len(scored))
    pearson_rows.append(row)
pearson_table = pd.DataFrame(pearson_rows)

print(
    'Table 10 — Pearson gate on EQ15 LOWESS (current |ρ| ≥ '
    f'{PEARSON_THRESH:.2f} → monotonic). '
    'Starred column is the current threshold. Flat excluded.'
)
show_full(pearson_table)

cost_rows = []
for thresh in PEARSON_SWEEP:
    scored = sens[sens['lowess_shape'] != 'flat']
    pred_nm = scored['ls_abs_pearson'] < thresh
    exp_nm = scored['family_id'].isin(EXPECTED_NONMONO)
    exp_mo = scored['family_id'].isin(EXPECTED_MONO)
    n_nm = int(exp_nm.sum())
    n_mo = int(exp_mo.sum())
    tpr = float(pred_nm[exp_nm].mean()) if n_nm else np.nan
    fpr = float(pred_nm[exp_mo].mean()) if n_mo else np.nan
    cost_rows.append({
        '|ρ| threshold': thresh,
        'Current': 'yes' if abs(thresh - PEARSON_THRESH) < 1e-12 else '',
        'Expected non-mono TPR': '' if pd.isna(tpr) else f'{tpr:.1%}',
        'Expected mono FPR': '' if pd.isna(fpr) else f'{fpr:.1%}',
        'F07 saturation → non-mono': _pct_fmt(
            int(((scored.family_id == 'F07') & pred_nm).sum()),
            int((scored.family_id == 'F07').sum()),
        ),
        'F22 oscillation → non-mono': _pct_fmt(
            int(((scored.family_id == 'F22') & pred_nm).sum()),
            int((scored.family_id == 'F22').sum()),
        ),
        'F25 multi-regime → non-mono': _pct_fmt(
            int(((scored.family_id == 'F25') & pred_nm).sum()),
            int((scored.family_id == 'F25').sum()),
        ),
    })
print(
    '\nTable 11 — Pearson sweep scored against expected mean-shape. '
    'Raising the gate recovers Oscillation only by also moving Saturation '
    'and F25 into non-monotonic.'
)
show_full(pd.DataFrame(cost_rows))

# --- R2 strength gates ---
r2_weak_rows = []
for family_id in _family_block():
    group = sens[sens['family_id'] == family_id]
    row = {
        'Family': family_id,
        'Name': FAMILY_NAMES[family_id],
        'N': len(group),
        'Median R²': float(group['r2'].median()),
    }
    for thresh in R2_WEAK_SWEEP:
        n_w = int((group['r2'] < thresh).sum())
        mark = ' *' if abs(thresh - R2_WEAK) < 1e-12 else ''
        row[f'Weak R²<{thresh:.2f}{mark}'] = _pct_fmt(n_w, len(group))
    r2_weak_rows.append(row)
print(
    f'\nTable 12 — R² Weak gate (current R² < {R2_WEAK:.2f}). '
    'Starred column is the current threshold.'
)
show_full(pd.DataFrame(r2_weak_rows).style.format({'Median R²': '{:.3f}'}))

r2_det_rows = []
for family_id in _family_block():
    group = sens[sens['family_id'] == family_id]
    row = {
        'Family': family_id,
        'Name': FAMILY_NAMES[family_id],
        'N': len(group),
    }
    for thresh in R2_DETECT_SWEEP:
        n_d = int((group['r2'] >= thresh).sum())
        mark = ' *' if abs(thresh - R2_DETECT) < 1e-12 else ''
        row[f'Detectable R²≥{thresh:.2f}{mark}'] = _pct_fmt(n_d, len(group))
    r2_det_rows.append(row)
print(
    f'\nTable 13 — R² Detectable gate (current R² ≥ {R2_DETECT:.2f}). '
    'Starred column is the current threshold.'
)
show_full(pd.DataFrame(r2_det_rows))

# --- Close-up: Spike / Oscillation / F25, and extra MIC ---
print('\nTable 14 — Current labels for Spike, Oscillation, and F25')
close = []
for family_id in ['F18', 'F19', 'F22', 'F25']:
    group = sens[sens['family_id'] == family_id]
    close.append({
        'Family': family_id,
        'Name': FAMILY_NAMES[family_id],
        'N': len(group),
        'Median R²': float(group['r2'].median()),
        'Median |ρ|': float(group['ls_abs_pearson'].median()),
        'Median n_tp': float(group['ls_n_tp'].median()),
        'n_tp = 0': _pct_fmt(int((group['ls_n_tp'] == 0).sum()), len(group)),
        'Non-monotonic': _pct_fmt(
            int((group['lowess_shape'] == 'nonmonotonic').sum()), len(group),
        ),
        'inverted_U': _pct_fmt(
            int((group['lowess_shape2'] == 'inverted_U').sum()), len(group),
        ),
        'U_shaped': _pct_fmt(
            int((group['lowess_shape2'] == 'U_shaped').sum()), len(group),
        ),
        'complex / S_or_N': _pct_fmt(
            int(group['lowess_shape2'].isin(['complex', 'S_or_N']).sum()),
            len(group),
        ),
    })
show_full(pd.DataFrame(close).style.format({
    'Median R²': '{:.3f}', 'Median |ρ|': '{:.3f}', 'Median n_tp': '{:.1f}',
}))

pearson_table.to_csv(OUTPUT_DIR / 'lowess_pearson_threshold_sensitivity.csv', index=False)
pd.DataFrame(r2_weak_rows).to_csv(
    OUTPUT_DIR / 'lowess_r2_weak_threshold_sensitivity.csv', index=False,
)
print(f'\nWrote Pearson/R² sensitivity CSVs under {OUTPUT_DIR}')

Table 10 — Pearson gate on EQ15 LOWESS (current |ρ| ≥ 0.60 → monotonic). Starred column is the current threshold. Flat excluded.


,Family,Name,N,N scored,Expected,Non-mono |ρ|<0.50,Non-mono |ρ|<0.60 *,Non-mono |ρ|<0.70,Non-mono |ρ|<0.80
0,F23,Two Lines,681,681,Unscored,81 (11.9%),132 (19.4%),177 (26.0%),206 (30.2%)
1,F24,Line and Parabola,681,681,Unscored,131 (19.2%),152 (22.3%),225 (33.0%),288 (42.3%)
2,F01,Linear positive,681,663,Monotonic,93 (14.0%),108 (16.3%),124 (18.7%),146 (22.0%)
3,F03,Power convex positive,681,681,Monotonic,5 (0.7%),17 (2.5%),43 (6.3%),80 (11.7%)
4,F05,Power concave positive,681,681,Monotonic,22 (3.2%),45 (6.6%),67 (9.8%),96 (14.1%)
5,F07,Saturation positive,681,681,Monotonic,41 (6.0%),105 (15.4%),298 (43.8%),337 (49.5%)
6,F13,S-curve positive,681,681,Monotonic,68 (10.0%),83 (12.2%),110 (16.2%),141 (20.7%)
7,F15,Threshold positive,681,681,Monotonic,34 (5.0%),69 (10.1%),102 (15.0%),142 (20.9%)
8,F18,Quadratic valley (U),681,681,Non-monotonic,681 (100.0%),681 (100.0%),681 (100.0%),681 (100.0%)
9,F19,Spike,681,671,Non-monotonic,630 (93.9%),671 (100.0%),671 (100.0%),671 (100.0%)



Table 11 — Pearson sweep scored against expected mean-shape. Raising the gate recovers Oscillation only by also moving Saturation and F25 into non-monotonic.


,|ρ| threshold,Current,Expected non-mono TPR,Expected mono FPR,F07 saturation → non-mono,F22 oscillation → non-mono,F25 multi-regime → non-mono
0,0.5,,83.7%,6.4%,41 (6.0%),390 (57.3%),77 (11.3%)
1,0.6,yes,88.8%,10.2%,105 (15.4%),454 (66.7%),108 (15.9%)
2,0.7,,90.5%,17.1%,298 (43.8%),488 (71.7%),564 (82.8%)
3,0.8,,100.0%,22.1%,337 (49.5%),681 (100.0%),604 (88.7%)



Table 12 — R² Weak gate (current R² < 0.35). Starred column is the current threshold.


...


Table 13 — R² Detectable gate (current R² ≥ 0.60). Starred column is the current threshold.


,Family,Name,N,Detectable R²≥0.50,Detectable R²≥0.55,Detectable R²≥0.60 *,Detectable R²≥0.70
0,F23,Two Lines,681,108 (15.9%),103 (15.1%),96 (14.1%),72 (10.6%)
1,F24,Line and Parabola,681,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
2,F01,Linear positive,681,362 (53.2%),353 (51.8%),344 (50.5%),324 (47.6%)
3,F03,Power convex positive,681,362 (53.2%),353 (51.8%),344 (50.5%),323 (47.4%)
4,F05,Power concave positive,681,371 (54.5%),361 (53.0%),351 (51.5%),331 (48.6%)
5,F07,Saturation positive,681,368 (54.0%),359 (52.7%),349 (51.2%),329 (48.3%)
6,F13,S-curve positive,681,368 (54.0%),359 (52.7%),349 (51.2%),329 (48.3%)
7,F15,Threshold positive,681,367 (53.9%),358 (52.6%),348 (51.1%),327 (48.0%)
8,F18,Quadratic valley (U),681,367 (53.9%),357 (52.4%),347 (51.0%),327 (48.0%)
9,F19,Spike,681,125 (18.4%),121 (17.8%),117 (17.2%),30 (4.4%)



Table 14 — Current labels for Spike, Oscillation, and F25


,Family,Name,N,Median R²,Median |ρ|,Median n_tp,n_tp = 0,Non-monotonic,inverted_U,U_shaped,complex / S_or_N
0,F18,Quadratic valley (U),681,0.633,0.021,1.0,0 (0.0%),681 (100.0%),0 (0.0%),434 (63.7%),247 (36.3%)
1,F19,Spike,681,0.240,0.074,2.0,10 (1.5%),671 (98.5%),292 (42.9%),0 (0.0%),379 (55.7%)
2,F22,Oscillation,681,0.596,0.462,4.0,0 (0.0%),454 (66.7%),0 (0.0%),0 (0.0%),454 (66.7%)
3,F25,Multi-regime threshold,681,0.289,0.614,4.0,31 (4.6%),108 (15.9%),28 (4.1%),0 (0.0%),80 (11.7%)



Wrote Pearson/R² sensitivity CSVs under /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/F/output/F_S3.3_caseA_classifier_test/constant/even


## LOWESS examples by synthetic family

For each family, the next cell selects five cases spread across the available low-to-moderate SNR range. The default cap is SNR ≤ 2.0, which is far below the family medians in this dataset. The scatter shows the synthetic observations and the magenta line is the cached LOWESS fit used by the classifier.


In [8]:
# Five low/moderate-SNR LOWESS examples per family
import matplotlib.pyplot as plt

LOWESS_EXAMPLES_PER_FAMILY = 5
LOWESS_EXAMPLE_SNR_MAX = 2.0  # 可调：限制示例的最大 nominal SNR
LOWESS_EXAMPLE_MAX_SCATTER = 3000
LOWESS_EXAMPLE_SEED = 42
LOWESS_EXAMPLE_DIR = OUTPUT_DIR / 'lowess_family_examples'
LOWESS_EXAMPLE_DIR.mkdir(parents=True, exist_ok=True)

curve_examples = pd.read_parquet(
    OUTPUT_DIR / 'lowess_mean_response_curves.parquet'
)
curve_examples['_case_key'] = curve_examples['case_id'].astype(str)
curve_lookup_examples = {
    str(case_id): (
        np.asarray(xs, dtype=float), np.asarray(ys, dtype=float)
    )
    for case_id, xs, ys in zip(
        curve_examples['case_id'],
        curve_examples['x_lowess'],
        curve_examples['y_lowess'],
    )
}

plot_meta = lowess_eval.copy()
plot_meta['_case_key'] = plot_meta['case_id'].astype(str)
point_index_lookup = (
    data.assign(_case_key=data['case_id'].astype(str))
        .drop_duplicates('_case_key')
        .set_index('_case_key')['point_index']
)
plot_meta['point_index'] = plot_meta['_case_key'].map(point_index_lookup)


def pick_low_snr_examples(group, n=5, snr_max=2.0):
    candidates = group[np.isfinite(group['nominal_snr'])].copy()
    below_cap = candidates[candidates['nominal_snr'] <= snr_max]
    if len(below_cap) >= n:
        candidates = below_cap
    else:
        candidates = candidates.nsmallest(n, 'nominal_snr')
    candidates = candidates.sort_values('nominal_snr')
    if len(candidates) <= n:
        return candidates
    positions = np.linspace(0, len(candidates) - 1, n).round().astype(int)
    return candidates.iloc[np.unique(positions)]


rng = np.random.default_rng(LOWESS_EXAMPLE_SEED)
selected_example_rows = []
for family_id in FAMILY_FILTER:
    family_group = plot_meta[plot_meta['family_id'] == family_id]
    chosen = pick_low_snr_examples(
        family_group,
        n=LOWESS_EXAMPLES_PER_FAMILY,
        snr_max=LOWESS_EXAMPLE_SNR_MAX,
    )
    if chosen.empty:
        continue
    selected_example_rows.append(chosen)

    fig, axes = plt.subplots(
        1, len(chosen), figsize=(3.25 * len(chosen), 3.1), squeeze=False
    )
    axes = axes.ravel()
    for ax, row in zip(axes, chosen.itertuples(index=False)):
        point_index = int(row.point_index)
        x = np.asarray(x_all[point_index], dtype=float)
        y = np.asarray(y_all[point_index], dtype=float)
        finite = np.isfinite(x) & np.isfinite(y)
        x, y = x[finite], y[finite]
        if len(x) > LOWESS_EXAMPLE_MAX_SCATTER:
            keep = rng.choice(
                len(x), LOWESS_EXAMPLE_MAX_SCATTER, replace=False
            )
            x_show, y_show = x[keep], y[keep]
        else:
            x_show, y_show = x, y
        ax.scatter(
            x_show, y_show, s=7, color='#6b7280', alpha=0.18,
            linewidths=0, rasterized=True,
        )
        xs, ys = curve_lookup_examples[str(row.case_id)]
        ax.plot(xs, ys, color='#d81b60', linewidth=2.0, label='LOWESS')
        ax.axhline(0, color='#9ca3af', linewidth=0.7, alpha=0.6)
        rho_text = (
            'na' if pd.isna(row.ls_pearson) else f'{row.ls_pearson:.2f}'
        )
        ax.set_title(
            f'SNR={row.nominal_snr:.2g}   R²={row.r2:.2f}\n'
            f'Pearson={rho_text}   {row.strength} / {row.primary_shape}',
            fontsize=9,
        )
        ax.set_xlabel('x')
        ax.grid(alpha=0.18, linewidth=0.6)
    axes[0].set_ylabel('y')
    fig.suptitle(
        f'{family_id}: {FAMILY_NAMES[family_id]} — low/moderate SNR examples',
        fontsize=12, y=1.02,
    )
    fig.tight_layout()
    save_path = LOWESS_EXAMPLE_DIR / f'{family_id}_lowess_examples.png'
    fig.savefig(save_path, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)

selected_examples = pd.concat(selected_example_rows, ignore_index=True)
selected_examples[
    ['family_id', 'case_id', 'nominal_snr', 'r2', 'ls_pearson',
     'strength', 'primary_shape']
].to_csv(LOWESS_EXAMPLE_DIR / 'selected_lowess_examples.csv', index=False)
print(f'Saved family example figures to {LOWESS_EXAMPLE_DIR}')


/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_4529/604035639.py:105: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved family example figures to /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/F/output/F_S3.3_caseA_classifier_test/constant/even/lowess_family_examples


In [9]:
# F21 / F22 / F25: scatter + LOWESS + Pearson — why |P|>=0.6 fails
# Each row = one family, 6 columns = 6 representative cases (R²≥0.35)
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess

FOCUS_FAMILIES = ['F21', 'F22', 'F25']
N_COLS = 6
FRAC_PLOT = 0.25
MAX_SCATTER = 3000
SEED = 123

rng_plot = np.random.default_rng(SEED)

# Use the frac=0.25 features we already have, or recompute on the fly
focus_eval = lowess_eval[
    lowess_eval['family_id'].isin(FOCUS_FAMILIES)
    & pd.to_numeric(lowess_eval['r2'], errors='coerce').ge(0.35)
].copy()
focus_eval['point_index'] = focus_eval['case_id'].map(
    data.drop_duplicates('case_id').set_index('case_id')['point_index']
)

fig, axes = plt.subplots(
    len(FOCUS_FAMILIES), N_COLS,
    figsize=(3.4 * N_COLS, 3.3 * len(FOCUS_FAMILIES)),
    squeeze=False,
)

for row_i, fam in enumerate(FOCUS_FAMILIES):
    fam_data = focus_eval[focus_eval['family_id'] == fam].copy()
    fam_data = fam_data.sort_values('ls_abs_pearson')

    # Pick N_COLS cases spread across the |P| range
    if len(fam_data) <= N_COLS:
        chosen = fam_data
    else:
        positions = np.linspace(0, len(fam_data) - 1, N_COLS).round().astype(int)
        chosen = fam_data.iloc[np.unique(positions)]
        if len(chosen) < N_COLS:
            chosen = fam_data.iloc[:N_COLS]

    for col_i, (_, case) in enumerate(chosen.iterrows()):
        if col_i >= N_COLS:
            break
        ax = axes[row_i, col_i]
        pi = int(case['point_index'])
        x = np.asarray(x_all[pi], dtype=float)
        y = np.asarray(y_all[pi], dtype=float)
        fin = np.isfinite(x) & np.isfinite(y)
        x, y = x[fin], y[fin]

        # Subsample for display
        if len(x) > MAX_SCATTER:
            keep = rng_plot.choice(len(x), MAX_SCATTER, replace=False)
            xs_show, ys_show = x[keep], y[keep]
        else:
            xs_show, ys_show = x, y

        ax.scatter(xs_show, ys_show, s=6, color='#6b7280', alpha=0.15,
                   linewidths=0, rasterized=True)

        # Fit LOWESS at frac=0.25
        result = sm_lowess(y, x, frac=FRAC_PLOT, it=3, return_sorted=True)
        ax.plot(result[:, 0], result[:, 1], color='#d81b60', linewidth=2.0)

        p_val = float(case['ls_abs_pearson']) if pd.notna(case['ls_abs_pearson']) else float('nan')
        r2_val = float(case['r2'])
        tp_val = int(case['ls_n_tp']) if pd.notna(case['ls_n_tp']) else 0

        ax.set_title(
            f"|P|={p_val:.3f}  R²={r2_val:.2f}  TP={tp_val}",
            fontsize=9, fontweight='bold',
            color='#c62828' if p_val >= 0.6 else '#2e7d32',
        )
        ax.grid(alpha=0.18, linewidth=0.6)
        if col_i == 0:
            ax.set_ylabel(f'{fam}\n{FAMILY_NAMES[fam]}', fontsize=10, fontweight='bold')

    # Fill remaining columns if fewer cases
    for col_i in range(len(chosen), N_COLS):
        axes[row_i, col_i].set_visible(False)

fig.suptitle(
    f'F21 / F22 / F25: NonMono families where |P|≥0.6 fails\n'
    f'(frac={FRAC_PLOT}, R²≥0.35 — red title = |P|≥0.6 misclassified as Mono)',
    fontsize=13, y=1.02,
)
fig.tight_layout()
fig.savefig(
    LOWESS_EXAMPLE_DIR / 'F21_F22_F25_pearson_failure.png',
    dpi=180, bbox_inches='tight',
)
plt.show()
plt.close(fig)
print(f'Saved to {LOWESS_EXAMPLE_DIR / "F21_F22_F25_pearson_failure.png"}')


Saved to /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/F/output/F_S3.3_caseA_classifier_test/constant/even/lowess_family_examples/F21_F22_F25_pearson_failure.png


/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_4529/3772971884.py:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
